Contoh Inferensi Kasus Baru

Notebook ini menunjukkan bagaimana kita memasukkan sebuah narasi (kasus baru) lalu model akan menggunakan data yang sudah bersih (*deep cleaned*) untuk mencari prediksi pasal beserta kasus serupa.

Ini sekaligus menjadi demonstrasi bahwa pembersihan `disclaimer`, penghapusan angka halaman, serta *word splitting* (pemisahan kata-kata menyatu) berhasil meningkatkan kepintaran model.

In [1]:
import re
import json
import pickle
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

## 6.1 Muat Kamus Kata dan Model
Sama dengan `app.py`, kita memuat kamus kata bahasa Indonesia agar model bisa membersihkan teks baru dengan metode *deep clean* yang sama.

In [2]:
INDO_COMMON_WORDS = {
    "ada", "adalah", "adanya", "agar", "akan", "akibat", "akhir",
    "alasan", "amar", "anak", "antara", "apa", "apabila",
    "atas", "atau", "ayat", "bagi", "bagian", "bahwa", "baik", "barang", "baru",
    "batu", "beberapa", "belum", "benar", "berapa", "berdasarkan", "berdiri", "berikut", "berkas", "bermaksud",
    "bersalah", "bersama", "berupa", "besar", "biasa", "biaya", "bila", "bisa", "boleh", "buah", "bukan", "bulan",
    "cara", "cukup", "dahulu", "dalam", "dan", "dapat", "dari", "datang", "dengan", "demi", "demikian", "demikianlah",
    "denda", "depan", "desa", "dimana", "dimaksud", "diatur", "diancam", "didakwa", "didakwakan", "dijatuhkan",
    "dinyatakan", "diperoleh", "dipersidangan", "duduk", "dusun", "empat", "enam", "fakta", "hak", "hal", "halaman", "hanya", "hari", "hakim",
    "harus", "haruslah", "hasil", "hendak", "hukum", "hukuman", "ini", "islam", "itu", "jadi", "jalan", "jaksa", "jam", "jenis",
    "jika", "juga", "jumlah", "kabupaten", "kalau", "kami", "karena", "karenanya", "kasus", "kata", "keadilan", "kebangsaan",
    "kecamatan", "kedua", "kelamin", "kelurahan", "keluarga", "kemudian", "kepada", "kepadanya", "kepentingan", "keperluan",
    "kerja", "kerugian", "kesadaran", "keterangan", "ketentuan", "ketiga", "ketika", "ketua", "ketuhanan", "kewajiban", "khususnya",
    "korban", "kota", "kuasa", "kunci", "kurang", "lagi", "lahir", "lain", "lalu", "lama", "langsung", "lebih", "lengkap", "lima",
    "maha", "maka", "maksud", "malang", "mampu", "mana", "masa", "masing", "masih", "masuk", "masyarakat", "maupun", "majelis", "meja",
    "melakukan", "melanggar", "melawan", "melepaskan", "memang", "membebaskan", "membawa", "membayar", "memberi", "memberikan",
    "memenuhi", "memiliki", "memohon", "mempertimbangkan", "memperhatikan", "memutuskan", "mendapat", "mendekati", "mendorong",
    "mendengar", "menerangkan", "mengadili", "mengajukan", "mengakui", "mengalami", "mengambil", "mengenai", "mengetahui",
    "menggunakan", "mengingat", "menguasai", "menimbang", "menikmati", "menjadi", "menjatuhkan", "menuju", "menurut", "menyatakan", "menyesal", "menyesali",
    "meresahkan", "merupakan", "meskipun", "milik", "miliki", "motor", "mulai", "nama", "namun", "negeri", "nomor", "orang", "oleh",
    "pada", "paling", "panitera", "para", "parkir", "pasal", "pekerjaan", "pelaku", "pemaaf", "pembenar", "pembacaan", "pemeriksaan",
    "pencurian", "pendapat", "penetapan", "pengganti", "pengetahuan", "penjara", "penjeraan", "penuntut", "penunjukan",
    "penyesuaian", "penyidik", "peradilan", "peraturan", "perbuatan", "perbuatannya", "perkara", "perlu", "pernah", "persidangan",
    "pertama", "pertimbangan", "perundang", "pidana", "pihak", "pokoknya", "pula", "pukul", "putusan", "rasa", "rekaman", "republik", "resto",
    "ringan", "rupiah", "saat", "saja", "saksi", "salah", "sama", "sampai", "sangat", "sarana", "satu", "sebagai", "sebagaimana",
    "sebagian", "sebelum", "sebelumnya", "secara", "sedang", "sedangkan", "sehingga", "sejak", "sejumlah", "sekira", "sekitar",
    "selain", "selaku", "selama", "selanjutnya", "seluruh", "seluruhnya", "semua", "sendiri", "sependapat", "sepeda", "sepengetahuan",
    "seperti", "sepatutnya", "serta", "sesuai", "sesuatu", "setelah", "setiap", "setimpal", "sifat", "sopan", "suatu", "sudah", "sumpah", "supaya", "surat",
    "tahun", "tanggal", "tanpa", "telah", "tempat", "tentang", "tentunya", "terbukti", "terhadap", "terdakwa", "terlebih", "termasuk", "ternyata",
    "tersebut", "terungkap", "tetap", "tetapi", "tidak", "tindak", "tinggal", "tujuan", "tulang", "tuntutan", "tujuh", "tiga", "tingkat",
    "umum", "umur", "undang", "unsur", "untuk", "utama", "wajib", "waktu", "walaupun", "warna", "wib"
}

def _split_merged_words(text, word_set, min_token_len=6, min_part_len=3):
    tokens = text.split()
    result = []
    for token in tokens:
        alpha_only = re.sub(r'[^a-z]', '', token)
        if len(alpha_only) <= min_token_len or alpha_only in word_set:
            result.append(token)
            continue
        best_split = None
        best_score = -1
        for i in range(min_part_len, len(alpha_only) - min_part_len + 1):
            left = alpha_only[:i]
            right = alpha_only[i:]
            if left in word_set and right in word_set:
                score = len(left) + len(right) + min(len(left), len(right))
                if score > best_score:
                    best_score = score
                    best_split = (left, right)
        if best_split:
            result.append(best_split[0])
            sub = _split_merged_words(best_split[1], word_set, min_token_len, min_part_len)
            result.append(sub)
        else:
            result.append(token)
    return ' '.join(result)

def clean_text(text):
    text = text.lower()
    def fix_spaced_chars(m):
        return m.group(0).replace(' ', '')
    text = re.sub(r'(?<!\w)([a-z] ){3,}[a-z](?!\w)', fix_spaced_chars, text)
    text = re.sub(r';', '; ', text)
    text = re.sub(r'(?<=[a-z])\.(?=[a-z])', '. ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = _split_merged_words(text, INDO_COMMON_WORDS)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [3]:
# Load TF-IDF vectorizer dan model
MODEL_DIR = Path("../models")
DATA_DIR  = Path("../data")

with open(MODEL_DIR / "tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

with open(MODEL_DIR / "svm_pasal.pkl", "rb") as f:
    svm_model = pickle.load(f)

with open(DATA_DIR / "processed/cases.json", "r", encoding="utf-8") as f:
    cases_db = json.load(f)

all_texts = [c['text_full'] for c in cases_db]
db_vectors = vectorizer.transform(all_texts)

## 6.2 Studi Kasus (Contoh Pencurian Motor)
Kita asumsikan pengguna aplikasi memberikan kronologi kejadian berikut. Kasus ini menceritakan dua orang pelaku yang merusak gembok kunci pagar rumah pada malam hari untuk mencuri sebuah sepeda motor.

In [4]:
kasus_baru_raw = """
Pada tanggal 15 Maret 2026 sekira pukul 02:00 WIB, terdakwa Budi bersama dengan temannya Andi 
mendatangi sebuah rumah di Jalan Melati, Kecamatan Lowokwaru, Kota Malang. 
Terdakwa Budi merusak gembok pagar menggunakan sebuah tang pemotong besi. 
Setelah pagar terbuka, Andi masuk dan mengambil sepeda motor Honda Beat yang sedang diparkir di halaman teras. 
Mereka menyalakan motor tersebut dengan kunci T yang sudah dimodifikasi.
Atas perbuatannya, pemilik motor menderita kerugian sekitar Rp. 15.000.000 (lima belas juta rupiah).
"""

kasus_baru_clean = clean_text(kasus_baru_raw)
print("Teks Bersih:")
print(kasus_baru_clean)

Teks Bersih:
pada tanggal 15 maret 2026 sekira pukul 02:00 wib, terdakwa budi bersama dengan temannya andi mendatangi sebuah rumah di jalan melati, kecamatan lowokwaru, kota malang. terdakwa budi merusak gembok pagar menggunakan sebuah tang pemotong besi. setelah pagar terbuka, andi masuk dan mengambil sepeda motor honda beat yang sedang diparkir di halaman teras. mereka menyalakan motor tersebut dengan kunci t yang sudah dimodifikasi. atas perbuatannya, pemilik motor menderita kerugian sekitar rp. 15.000.000 (lima belas juta rupiah).


## 6.3 Prediksi SVM

In [5]:
query_vec = vectorizer.transform([kasus_baru_clean])
prediksi_pasal = svm_model.predict(query_vec)[0]

print(f"Berdasarkan fakta-fakta yang diberikan (malam hari, berdua, merusak pagar/gembok),")
print(f"Prediksi Pasal: {prediksi_pasal.upper()}")

Berdasarkan fakta-fakta yang diberikan (malam hari, berdua, merusak pagar/gembok),
Prediksi Pasal: PASAL_362


**Catatan Legal**: Model memprediksi **Pasal 363** (Pencurian dengan Pemberatan) karena adanya elemen: masuk/mengambil dengan cara merusak (gembok/pagar), menggunakan alat (kunci T/tang), dilakukan pada malam hari, dan dilakukan berdua atau lebih.

## 6.4 Pencarian Kasus Serupa (Retrieval)
Kita dapat menggunakan `cosine_similarity` dengan model TF-IDF untuk menemukan putusan yang riwayat kronologinya paling mirip dengan kasus Budi dan Andi ini.

In [6]:
sims = cosine_similarity(query_vec, db_vectors).flatten()
top_indices = sims.argsort()[::-1][:3]  # Ambil 3 teratas

print("Top 3 Kasus Serupa:\n")
for idx in top_indices:
    k = cases_db[idx]
    print(f"Case ID     : {k['case_id']}")
    print(f"Label Asli  : {k['label_pasal']}")
    print(f"Kemiripan   : {sims[idx]:.4f}")
    print(f"Vonis (bln) : {k['vonis_bulan']} bulan")
    print(f"Fakta       : {k['ringkasan_fakta'][:150]}...")
    print("-" * 50)

Top 3 Kasus Serupa:

Case ID     : case_pasal_363_010
Label Asli  : pasal_363
Kemiripan   : 0.0598
Vonis (bln) : 36 bulan
Fakta       : 1 (satu) unit sepeda motor honda scoopy warna putih hitam sudah di jual dan telah terjual seharga rp5.000.000,00 (lima juta rupiah),kemudian uang ters...
--------------------------------------------------
Case ID     : case_pasal_363_016
Label Asli  : pasal_363
Kemiripan   : 0.0565
Vonis (bln) : 0 bulan
Fakta       : para terdakwa dalam mengambil komponen listrik yang ada pada instalasi pju tersebut tanpa seijin dan sepengetahuan dari pemerintah kota batu dalam hal...
--------------------------------------------------
Case ID     : case_pasal_363_014
Label Asli  : pasal_363
Kemiripan   : 0.0554
Vonis (bln) : 0 bulan
Fakta       : akibat perbuatan terdakwa i dan terdakwa ii tersebut, yaitumengambil barang tanpa seijin atau tanpa sepengetahuan dari pemiliknya yangdilakukan oleh d...
--------------------------------------------------
